In [1]:
# from starter import rag



A trace is the end-to-end story of a single request as it moves through your system. For us, it's one RAG call.

A span is one operation within a trace. A trace is made of one or more spans, organized as a tree. Each span has a name, a start and end time, and a set of attributes. For us we will have one span inside the trace, but for agents one trace will have multiple spans.

Attributes are key-value pairs attached to a span - anything you want to record, like the number of tokens used or the cost of a call.

When a span finishes - meaning the code block it wraps completes - the SDK hands it to a span processor, which forwards it to an exporter. The exporter decides where the span goes: to the console, to a file, to a database, or to a remote collector. We will see all of this in practice in the questions below.

We start with the ConsoleSpanExporter, which prints each finished span to the terminal so we can see what OTel captures:

In [1]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

# TracerProvider() creates the SDK's central configuration object. 
# It owns the span processors and decides how spans are built.
provider = TracerProvider()
provider.add_span_processor(
    # SimpleSpanProcessor(ConsoleSpanExporter()) wires a processor that forwards every finished span 
    # to the console exporter, one at a time. "Simple" means synchronous and immediate - good for development.
    SimpleSpanProcessor(ConsoleSpanExporter())
)
# trace.set_tracer_provider(provider) registers the provider globally, 
# so every call to trace.get_tracer(...) returns a tracer backed by it.
trace.set_tracer_provider(provider)
# returns a Tracer we use to create spans. The string is just a label for the instrumentation scope 
# - it identifies which part of the code produced the spans.
tracer = trace.get_tracer("llm-zoomcamp") # llm-zoomcap is identifying label

In [3]:
# start_as_current_span creates a new span and makes it the "current" span for the duration of the with block. 
# Any code inside the block - including other calls to start_as_current_span - becomes a child of this span. 
# When the block exits, the span ends automatically.
# with tracer.start_as_current_span("my_operation") as span:
#     # your code here
#     span.set_attribute("my_key", "my_value")

<h3> First Question </h3>

In [4]:
from rag_helper import RAGBase

In [5]:
class RAGTraced(RAGBase):

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search"):
            return super().search(
                query,
                num_results=num_results
            )
    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            usage = response.usage
            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)
            # input_cost = usage.input_tokens * INPUT_PRICE_PER_TOKEN
            # output_cost = usage.output_tokens * OUTPUT_PRICE_PER_TOKEN
            # cost = input_cost + output_cost

            # span.set_attribute("cost", cost)
            return response
    def rag(self, query):
         with tracer.start_as_current_span("rag"):
            return super().rag(
                query
            )

In [6]:
from starter import index, client

traced_rag = RAGTraced(
    index=index,
    llm_client=client
)

In [8]:
# Wrap the rag() method so each call produces a span. 
# The simplest way is to create a RAGTraced subclass of RAGBase that wraps 
# rag(), search(), and llm() each in their own span.

query1 = "How does the agentic loop keep calling the model until it stops?"

answer = traced_rag.rag(query1)

print(answer)


{
    "name": "search",
    "context": {
        "trace_id": "0xd8f60e98918275b1335c3b88ea7318c8",
        "span_id": "0xae414933c1519903",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x7249b3ee9570885f",
    "start_time": "2026-07-20T19:34:17.851470Z",
    "end_time": "2026-07-20T19:34:17.854752Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "2ada158f-217d-4b80-a6ae-e90cafb47c31",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xd8f60e98918275b1335c3b88ea7318c8",
        "span_id": "0xc790373b23cf0e5a",
        "trace_state": "[]"
    },
    "kind": "SpanKind

<h3> Question 1 Answer: 3 </h3>

<h2> Question 2 </h2>

question 2 ANSWER: input_oken usage is 7111 so is about 7000

<h2> Question 3 </h2>

Question 3 Answer: 500–2000 ms (About 1,336 ms = 1.34 seconds from query payload)

<h2> Question 4 </h2>

In [9]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [10]:
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

In [11]:
answer = traced_rag.rag(query1)

print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0x06b166afb50a36549809ca03c3d25005",
        "span_id": "0xc4bb446b61ea6702",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x6d790a4ae78118ff",
    "start_time": "2026-07-20T19:39:48.848514Z",
    "end_time": "2026-07-20T19:39:48.852070Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "2ada158f-217d-4b80-a6ae-e90cafb47c31",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x06b166afb50a36549809ca03c3d25005",
        "span_id": "0xabda02c580d4c4bc",
        "trace_state": "[]"
    },
    "kind": "SpanKind

In [12]:
import sqlite3

conn = sqlite3.connect("traces.db")

rows = conn.execute("""
    SELECT DISTINCT name
    FROM spans
""").fetchall()

print(rows)
conn.close()

[('search',), ('llm',), ('rag',)]


Question 4 Answer: [('search',), ('llm',), ('rag',)]

<h2> Question 5 </h2>

In [19]:
answer = traced_rag.rag(query1)

{
    "name": "search",
    "context": {
        "trace_id": "0x2d6b8d92e73ea5131c87e0fb7ab63738",
        "span_id": "0xfb734bc7b1afa496",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xb12b0a96f78c63e9",
    "start_time": "2026-07-20T19:48:03.525817Z",
    "end_time": "2026-07-20T19:48:03.528155Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "2ada158f-217d-4b80-a6ae-e90cafb47c31",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x2d6b8d92e73ea5131c87e0fb7ab63738",
        "span_id": "0x353cf9aeeb81fb42",
        "trace_state": "[]"
    },
    "kind": "SpanKind

In [15]:
import sqlite3

conn = sqlite3.connect("traces.db")

rows = conn.execute("""
    SELECT
        name,
        SUM(end_time - start_time) / 1000000 AS total_duration_ms
    FROM spans
    WHERE name != 'rag'
    GROUP BY name
    ORDER BY total_duration_ms DESC
""").fetchall()

for name, duration in rows:
    print(f"{name}: {duration:.2f} ms")

conn.close()

llm: 4376.00 ms
search: 7.00 ms


Question 5 Answer: llm

<h2> Question 6 </h2>

In [16]:
import pandas as pd

In [20]:
conn = sqlite3.connect("traces.db")

df = pd.read_sql_query("""
    SELECT name, input_tokens
    FROM spans
    WHERE name = 'llm'
""", conn)

df

,name,input_tokens
0,llm,7111
1,llm,7111
2,llm,7111
3,llm,7111


In [21]:
tokens = df["input_tokens"]

minimum = tokens.min()
maximum = tokens.max()

variation_percent = (maximum - minimum) / minimum * 100

print("Input tokens:", tokens.tolist())
print("Minimum:", minimum)
print("Maximum:", maximum)
print(f"Variation: {variation_percent:.2f}%")

Input tokens: [7111, 7111, 7111, 7111]
Minimum: 7111
Maximum: 7111
Variation: 0.00%


Question 6 Answer: They are identical